<a href="https://colab.research.google.com/github/rajak-abh/algo-lab/blob/main/Lab_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Travelling Salesman Problem using Dynamic Programming.
Problem Description:
Given a set of cities and distances, find the shortest possible route that visits each city once and returns to the starting city.

#Theory
TSP uses dynamic programming with bit masking. Store results of visited subsets to reduce repeated work.

## Algorithm
1. Start from source city
2. Mark visited cities using bit mask
3. Try all unvisited cities
4. Choose minimum cost path
5. Return to source city

In [9]:
def tsp(graph):
    n = len(graph)
    dp = [[-1]*(1<<n) for _ in range(n)]
    path_choices = [[-1]*(1<<n) for _ in range(n)] # To store the next city in the optimal path

    def visit(city, mask):
        # Base case: if all cities are visited, return cost to go back to starting city (0)
        if mask == (1<<n) - 1:
            return graph[city][0]

        # If already computed, return the stored result
        if dp[city][mask] != -1:
            return dp[city][mask]

        ans = float('inf')
        best_next_city = -1

        # Try all unvisited cities
        for next_city_idx in range(n):
            # Skip if it's the current city or if it has already been visited
            if next_city_idx == city or (mask & (1<<next_city_idx)):
                continue

            cost_to_next = graph[city][next_city_idx]
            new_cost = cost_to_next + visit(next_city_idx, mask | (1<<next_city_idx))

            if new_cost < ans:
                ans = new_cost
                best_next_city = next_city_idx

        dp[city][mask] = ans
        path_choices[city][mask] = best_next_city # Store the chosen next city for path reconstruction
        return ans

    # Start the TSP from city 0 with only city 0 visited in the mask
    min_cost = visit(0, 1)

    # Reconstruct the path
    path = [0] # Start path from city 0
    current_city = 0
    current_mask = 1

    # We need to make n-1 transitions to visit all other n-1 cities
    for _ in range(n - 1):
        next_c = path_choices[current_city][current_mask]
        if next_c == -1: # Should not happen if a valid path exists
            print(f"Warning: Could not reconstruct full path from city {current_city} with mask {bin(current_mask)}")
            break
        path.append(next_c)
        current_mask |= (1 << next_c) # Mark next_c as visited
        current_city = next_c
    path.append(0) # Return to the starting city to complete the cycle

    return min_cost, path

In [10]:
graph = [
[0, 10, 15, 20],
[10, 0, 35, 25],
[15, 35, 0, 30],
[20, 25, 30, 0]
]

min_cost, path = tsp(graph)
print("Minimum travelling cost:", min_cost)
print("Path:", path)

Minimum travelling cost: 80
Path: [0, 1, 3, 2, 0]


# Exercise
1. Print path of cities visited
2. Take graph input from user
3. Test with different number of cities

### Take graph input from user

In [11]:
def get_user_graph():
    while True:
        try:
            n = int(input("Enter the number of cities: "))
            if n <= 1:
                print("Number of cities must be greater than 1.")
                continue
            break
        except ValueError:
            print("Invalid input. Please enter an integer.")

    print("Enter the adjacency matrix (distances between cities).")
    print("Enter each row as space-separated numbers. For example: 0 10 15 20")

    graph = []
    for i in range(n):
        while True:
            try:
                row_str = input(f"Enter row {i} (n numbers): ")
                row = list(map(int, row_str.split()))
                if len(row) != n:
                    print(f"Expected {n} numbers, but got {len(row)}. Please re-enter.")
                    continue
                graph.append(row)
                break
            except ValueError:
                print("Invalid input. Please enter space-separated integers.")


    is_valid = True
    for i in range(n):
        if graph[i][i] != 0:
            print(f"Warning: Diagonal element graph[{i}][{i}] is not 0. TSP assumes distance from city to itself is 0.")

        for j in range(n):
            if graph[i][j] != graph[j][i]:
                print(f"Warning: Matrix is not symmetric. graph[{i}][{j}] ({graph[i][j]}) != graph[{j}][{i}] ({graph[j][i]}). Using graph[{i}][{j}].")


    return graph

user_graph = get_user_graph()
min_cost_user, path_user = tsp(user_graph)
print("\nMinimum travelling cost for user graph:", min_cost_user)
print("Path for user graph:", path_user)

Enter the number of cities: 5
Enter the adjacency matrix (distances between cities).
Enter each row as space-separated numbers. For example: 0 10 15 20
Enter row 0 (n numbers): 5 6 7 8  9
Enter row 1 (n numbers): 1 2 3 4 5
Enter row 2 (n numbers): 1 2 3 4  6
Enter row 3 (n numbers):  5 6 7 8 9 
Enter row 4 (n numbers):  7 8 9 0 0

Minimum travelling cost for user graph: 19
Path for user graph: [0, 1, 4, 3, 2, 0]
